In [ ]:
# CELL 1: Imports and Basic Functions
# -*- coding: utf-8 -*-
"""
Improved Backpropagation Neural Network Implementation
- Clean architecture with proper function organization
- Added learning rate for better convergence control
- Improved gradient calculations and weight updates
- Better error tracking and visualization
"""

import numpy as np
import matplotlib.pyplot as plt

def sigmoid(x):
    """
    Sigmoid activation function
    Changed: Extracted into separate function for reusability (was inline calculations)
    """
    return 1 / (1 + np.exp(-np.clip(x, -500, 500)))  # Added clipping to prevent overflow

def sigmoid_derivative(x):
    """
    Derivative of sigmoid function
    Changed: Created dedicated function (was computed inline as x*(1-x))
    """
    return x * (1 - x)

In [ ]:
# CELL 2: Weight Initialization

def initialize_weights():
    """
    Initialize network weights and biases
    Changed: Centralized initialization (was scattered random assignments)
    Returns weights in a structured format
    """
    np.random.seed(42)  # For reproducible results
    weights = {
        'hidden': {
            'w00': np.random.uniform(-1, 1),  # Changed: uniform distribution instead of rand()
            'w01': np.random.uniform(-1, 1),
            'w10': np.random.uniform(-1, 1),
            'w11': np.random.uniform(-1, 1),
            'b1': 0.5  # Changed: smaller initial bias (was 1.0)
        },
        'output': {
            'w2': np.random.uniform(-1, 1),
            'w3': np.random.uniform(-1, 1),
            'b2': 0.5  # Changed: smaller initial bias (was 1.0)
        }
    }
    return weights

In [ ]:
# CELL 3: Forward Propagation Function

def forward_pass(x1, x2, weights):
    """
    Forward propagation through the network
    Changed: Organized into separate function (was inline in main loop)
    Returns all intermediate values for backpropagation
    """
    # Input to hidden layer
    z1 = weights['hidden']['w00'] * x1 + weights['hidden']['w01'] * x2 + weights['hidden']['b1']
    z2 = weights['hidden']['w10'] * x1 + weights['hidden']['w11'] * x2 + weights['hidden']['b1']

    # Hidden layer activations
    h1 = sigmoid(z1)
    h2 = sigmoid(z2)

    # Hidden to output layer
    z_out = weights['output']['w2'] * h1 + weights['output']['w3'] * h2 + weights['output']['b2']
    output = sigmoid(z_out)

    return {
        'z1': z1, 'z2': z2, 'h1': h1, 'h2': h2,
        'z_out': z_out, 'output': output
    }

In [ ]:
# CELL 4: Gradient Computation (Backpropagation)

def compute_gradients(x1, x2, target, forward_vals, weights):
    """
    Compute gradients for backpropagation
    Changed: Organized gradient computation into structured function (was scattered calculations)
    Uses chain rule systematically
    """
    # Extract forward pass values
    h1, h2, output = forward_vals['h1'], forward_vals['h2'], forward_vals['output']

    # Output layer gradients
    error = output - target
    output_delta = error * sigmoid_derivative(output)

    # Gradients for output layer weights
    grad_w2 = output_delta * h1
    grad_w3 = output_delta * h2
    grad_b2 = output_delta

    # Hidden layer gradients (backpropagated error)
    hidden_error_h1 = output_delta * weights['output']['w2']
    hidden_error_h2 = output_delta * weights['output']['w3']

    hidden_delta_h1 = hidden_error_h1 * sigmoid_derivative(h1)
    hidden_delta_h2 = hidden_error_h2 * sigmoid_derivative(h2)

    # Gradients for hidden layer weights
    grad_w00 = hidden_delta_h1 * x1
    grad_w01 = hidden_delta_h1 * x2  # Changed: corrected gradient (was using h2 derivative)
    grad_w10 = hidden_delta_h2 * x1  # Changed: corrected gradient (was using h1 derivative)
    grad_w11 = hidden_delta_h2 * x2
    grad_b1 = hidden_delta_h1 + hidden_delta_h2

    return {
        'hidden': {
            'w00': grad_w00, 'w01': grad_w01,
            'w10': grad_w10, 'w11': grad_w11,
            'b1': grad_b1
        },
        'output': {
            'w2': grad_w2, 'w3': grad_w3, 'b2': grad_b2
        }
    }


In [ ]:
# CELL 5: Weight Update Function

def update_weights(weights, gradients, learning_rate):
    """
    Update weights using computed gradients
    Changed: Added learning rate parameter (was fixed step size of 1)
    Organized weight updates systematically
    """
    # Update hidden layer weights
    for key in weights['hidden']:
        weights['hidden'][key] -= learning_rate * gradients['hidden'][key]

    # Update output layer weights
    for key in weights['output']:
        weights['output'][key] -= learning_rate * gradients['output'][key]

In [ ]:
# CELL 6: Training Function

def train_network():
    """
    Main training function
    Changed: Organized training loop with better structure and monitoring
    """
    # Training data - single sample (can be extended to multiple samples)
    x1, x2 = np.array([0]), np.array([0])
    target = np.array([0])

    # Initialize network
    weights = initialize_weights()

    # Training parameters
    learning_rate = 0.1  # Changed: Added learning rate control (was implicit 1.0)
    epochs = 10000

    # Training tracking
    error_history = []
    epoch_history = []

    print("Starting training...")
    print(f"Initial weights: Hidden layer: {weights['hidden']}")
    print(f"Initial weights: Output layer: {weights['output']}")

    # Training loop
    for epoch in range(epochs):
        # Forward pass
        forward_vals = forward_pass(x1, x2, weights)

        # Calculate error
        error = 0.5 * np.mean((forward_vals['output'] - target) ** 2)

        # Compute gradients
        gradients = compute_gradients(x1, x2, target, forward_vals, weights)

        # Update weights
        update_weights(weights, gradients, learning_rate)

        # Track progress
        if epoch % 1000 == 0:  # Changed: Added progress monitoring
            print(f"Epoch {epoch}: Error = {error:.6f}")

        error_history.append(error)
        epoch_history.append(epoch)

    return weights, error_history, epoch_history

In [ ]:
#CELL 7: Testing and Visualization Functions

def test_network(weights, x1, x2):
    """
    Test the trained network
    Changed: Created dedicated testing function (was inline code)
    """
    forward_vals = forward_pass(x1, x2, weights)
    return forward_vals['output']

def plot_training_progress(epoch_history, error_history):
    """
    Plot training error over epochs
    Changed: Improved plot formatting and labels
    """
    plt.figure(figsize=(10, 6))
    plt.plot(epoch_history, error_history, 'b-', linewidth=2)
    plt.xlabel('Epoch', fontsize=12)
    plt.ylabel('Mean Squared Error', fontsize=12)
    plt.title('Training Error Over Time', fontsize=14)
    plt.grid(True, alpha=0.3)
    plt.yscale('log')  # Changed: Log scale for better error visualization
    plt.show()

In [ ]:
# CELL 8: Main Execution

# Main execution
if __name__ == "__main__":
    # Train the network
    trained_weights, error_hist, epoch_hist = train_network()

    # Display final results
    print("\n" + "="*50)
    print("TRAINING COMPLETED")
    print("="*50)
    print(f"Final weights - Hidden layer: {trained_weights['hidden']}")
    print(f"Final weights - Output layer: {trained_weights['output']}")
    print(f"Final training error: {error_hist[-1]:.8f}")

    # Test the trained network
    print("\n" + "-"*30)
    print("TESTING PHASE")
    print("-"*30)

    # Test with training data
    test_x1, test_x2 = np.array([0]), np.array([0])
    prediction = test_network(trained_weights, test_x1, test_x2)
    print(f"Input: x1={test_x1[0]}, x2={test_x2[0]}")
    print(f"Predicted output: {prediction[0]:.6f}")
    print(f"Rounded prediction: {np.round(prediction)[0]}")

    # Plot training progress
    plot_training_progress(epoch_hist, error_hist)

Conclusion: This program successfully implemented and trained a simple two-layer backpropagation neural network for a single data point (0, 0) with a target output of 0. The training process, which ran for 10,000 epochs with a learning rate of 0.1, significantly reduced the mean squared error from an initial value of 0.084794 to a final value of 0.00010363. The trained network, with its adjusted weights and biases, was then tested with the same input (0, 0), resulting in a predicted output of approximately 0.014396, which rounds to 0, indicating that the network learned to correctly classify this specific input based on the provided training data.